# 10. Pass completion at sequence pace


### Purpose

Check whether teams complete passes while moving the puck at higher possession-sequence pace. The coaching question is: **can a team maintain passing execution at higher pace?**

### Expected outcome

A 5v5 pass-attempt table, pooled slower/middle/faster pace bands, complete/incomplete percentages, and context and team comparisons. Use the same sequence-before-event methodology as notebooks 07-09. Unlike notebook 05's five-second lookback, this analysis uses the full observed sequence before each release.


## 10.1. Setup and load processed data

Load augmented events, transition pace, and reference tables. Pool all sources and retain provenance in the attempt records. Pace is estimated puck movement before a release, not pass-flight speed or skating speed.


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

pd.set_option("display.max_columns", None)
ROOT_DIR = Path("..")
PROCESSED_DATA_DIR = ROOT_DIR / "data" / "processed"

pace_manifest = json.loads(
    (PROCESSED_DATA_DIR / "pace_export_manifest.json").read_text(encoding="utf-8")
)
events = pd.read_parquet(PROCESSED_DATA_DIR / "events_augmented.parquet").sort_values("event_id")
transitions = pd.read_parquet(PROCESSED_DATA_DIR / "pace_transitions.parquet")
teams = pd.read_parquet(PROCESSED_DATA_DIR / "teams.parquet")
games = pd.read_parquet(PROCESSED_DATA_DIR / "games.parquet")
players = pd.read_parquet(PROCESSED_DATA_DIR / "players.parquet")

display(pd.DataFrame([
    {"table": name, "rows": len(table), "columns": len(table.columns)}
    for name, table in {"events": events, "transitions": transitions,
                        "teams": teams, "games": games, "players": players}.items()
]))
print("Timing:", pace_manifest["timing"])


,table,rows,columns
0,events,86670,27
1,transitions,63298,25
2,teams,20,3
3,games,34,5
4,players,435,4


Timing: Recorded clocks preserved; speeds use notebook 02 modeled clocks. Fractions are estimated, not measured.


## 10.2. Define one record per 5v5 pass attempt

Use `Pass` and `Incomplete Pass` releases, counting each once. Generated reception rows are not additional attempts. A complete pass is defined by its normalized source event label.

Pair each release with its immediately following expanded reception, validating game, team, and outcome. Completed-pass endpoints are recorded receptions; incomplete-pass endpoints are intended targets. Length therefore describes endpoint displacement, not the actual traveled path of a failed pass. Retain zone, direct/indirect pass type, length, player, and team context.


In [2]:
passes = events.loc[events["event"].isin(["Pass", "Incomplete Pass"]) & events["is_5v5"]].copy()
passes["complete"] = passes["event"].eq("Pass")
passes["pass_type"] = passes["detail_1"].fillna("Unknown")
passes["situation"] = (
    passes["team_skaters"].astype(str) + "v" + passes["opponent_skaters"].astype(str)
)
passes["team_name"] = passes["team_id"].map(teams.set_index("team_id")["team_name"])
passes["player_name"] = passes["player_id"].map(players.set_index("player_id")["player_name"])
passes["game_date"] = passes["game_id"].map(games.set_index("game_id")["game_date"])

receptions = events.loc[events["event"].isin(["Pass Reception", "Incomplete Pass Reception"])].copy()
receptions["release_event_id"] = receptions["event_id"] - 1
passes = passes.merge(
    receptions[["release_event_id", "game_id", "team_id", "event", "x", "y"]],
    left_on="event_id", right_on="release_event_id", suffixes=("", "_target"),
    how="left", validate="one_to_one",
)
assert passes["release_event_id"].notna().all()
assert passes["game_id"].eq(passes["game_id_target"]).all()
assert passes["team_id"].eq(passes["team_id_target"]).all()
assert passes["complete"].eq(passes["event_target"].eq("Pass Reception")).all()
passes["pass_length_ft"] = np.hypot(passes["x_target"] - passes["x"], passes["y_target"] - passes["y"])
passes["length_band"] = pd.cut(
    passes["pass_length_ft"].astype(float), bins=[0, 20, 40, np.inf],
    labels=["0-20 ft", "20-40 ft", "Over 40 ft"], include_lowest=True,
)

passes["pass_outcome"] = np.where(passes["complete"], "Complete", "Incomplete")
display(passes.groupby("pass_outcome").size().rename("attempts").reset_index())


,pass_outcome,attempts
0,Complete,12682
1,Incomplete,5913


## 10.3. Calculate possession-sequence pace before the release

Pool eligible transition distance and modeled movement duration within the pass's inferred possession sequence, using end event IDs strictly before the release. Exclude the focal pass, its generated reception, movement ending at its release, and all later movement. Do not use full exported sequence pace, which can include the pass outcome.

There is no fixed lookback. Earlier same-time events use event order. Empty or zero-duration histories have unknown preceding pace and remain outside tertiles, with coverage reported. Possession sequences split at manpower changes, so the preceding sequence for a 5v5 release is also 5v5. Modeled timestamps remain estimates.


In [3]:
valid_movements = transitions.loc[transitions["exclusion_reason"].isna()].copy()
# Reuse existing possession sequences; trim each one at the attempt event.
movement_groups = {
    pid: group[["end_event_id", "distance_ft", "modeled_elapsed_seconds"]].to_numpy(dtype=float)
    for pid, group in valid_movements.groupby("possession_id", sort=False)
}
sequence_start = events.groupby("possession_id")["modeled_clock_seconds"].max()
pace_rows = []
for attempt in passes.itertuples(index=False):
    movements = movement_groups.get(attempt.possession_id)
    if movements is not None:
        movements = movements[movements[:, 0] < attempt.event_id]
    count = 0 if movements is None else len(movements)
    distance = 0.0 if movements is None else float(movements[:, 1].sum())
    duration = 0.0 if movements is None else float(movements[:, 2].sum())
    pace_rows.append({
        "event_id": attempt.event_id,
        "preceding_transitions": count,
        "preceding_distance_ft": distance,
        "preceding_movement_seconds": duration,
        "preceding_pace_ft_s": distance / duration if duration > 0 else np.nan,
        "available_history_seconds": float(
            sequence_start.loc[attempt.possession_id] - attempt.modeled_clock_seconds
        ),
        "pace_status": "eligible" if duration > 0 else "no_history" if count == 0 else "zero_duration",
    })
passes = passes.merge(pd.DataFrame(pace_rows), on="event_id", validate="one_to_one")
coverage = passes.groupby("pace_status").agg(
    passes=("event_id", "size"), outcome_rate=("complete", "mean")
).reset_index()
display(coverage)


,pace_status,passes,outcome_rate
0,eligible,11064,0.684382
1,no_history,7466,0.678007
2,zero_duration,65,0.738462


## 10.4. Define pooled pace tertiles

Define three equal-frequency pace groups from all eligible 5v5 passes: slower, middle, faster. Share boundaries across teams and context groups, keeping ties together. Duplicate quantile edges may reduce the number of bands. Sources are combined without adjustment, so competition and recording differences may influence pooled results.


In [4]:
analysis_passes = passes.loc[passes["pace_status"].eq("eligible")].copy()
bins, pace_edges = pd.qcut(analysis_passes["preceding_pace_ft_s"], q=3,
                          labels=False, retbins=True, duplicates="drop")
band_names = {1: "Slower", 2: "Middle", 3: "Faster"}
analysis_passes["pace_tertile"] = (bins + 1).astype("Int64")
analysis_passes["pace_band"] = analysis_passes["pace_tertile"].map(band_names)
pace_boundaries = pd.DataFrame({
    "pace_tertile": np.arange(1, len(pace_edges)),
    "lower_pace_ft_s": pace_edges[:-1], "upper_pace_ft_s": pace_edges[1:],
})
pace_boundaries["pace_band"] = pace_boundaries["pace_tertile"].map(band_names)
display(pace_boundaries.round(2))
print(f"Passes with measurable preceding pace: {len(analysis_passes)} of {len(passes)} 5v5 attempts")


,pace_tertile,lower_pace_ft_s,upper_pace_ft_s,pace_band
0,1,0.00,16.75,Slower
1,2,16.75,24.31,Middle
2,3,24.31,401.46,Faster


Passes with measurable preceding pace: 11064 of 18595 5v5 attempts


## 10.5. Compare complete and incomplete passes

Show complete/incomplete percentages as stacked bars with no uncertainty intervals. Retain counts, games, median pace, and mean endpoint length. A lower completion percentage at higher pace is a pattern to inspect, not proof that faster play caused failures.


In [5]:
PACE_ORDER = ["Slower", "Middle", "Faster"]
OUTCOME_ORDER = ["Complete", "Incomplete"]
OUTCOME_COLORS = {"Complete": "#2563eb", "Incomplete": "#ea580c"}

def summarize_completion(data, keys):
    summary = data.groupby(keys, observed=True, dropna=False).agg(
        attempts=("event_id", "size"), completed=("complete", "sum"),
        games=("game_id", "nunique"), median_pace_ft_s=("preceding_pace_ft_s", "median"),
        mean_pass_length_ft=("pass_length_ft", "mean"),
    ).reset_index()
    summary["incomplete"] = summary["attempts"] - summary["completed"]
    summary["Complete_pct"] = 100 * summary["completed"] / summary["attempts"]
    summary["Incomplete_pct"] = 100 * summary["incomplete"] / summary["attempts"]
    return summary

completion_summary = summarize_completion(analysis_passes, ["pace_band"])
display(completion_summary.round(2))
completion_plot = completion_summary.melt(
    id_vars=["pace_band", "attempts", "games", "median_pace_ft_s"],
    value_vars=["Complete_pct", "Incomplete_pct"], var_name="pass_outcome", value_name="outcome_pct",
)
completion_plot["pass_outcome"] = completion_plot["pass_outcome"].str.removesuffix("_pct")
completion_fig = px.bar(
    completion_plot, x="pace_band", y="outcome_pct", color="pass_outcome",
    category_orders={"pace_band": PACE_ORDER, "pass_outcome": OUTCOME_ORDER},
    color_discrete_map=OUTCOME_COLORS, hover_data=["attempts", "games", "median_pace_ft_s"],
    labels={"pace_band": "Preceding sequence pace", "outcome_pct": "Pass attempts (%)", "pass_outcome": "Outcome"},
    title="5v5 pass completion at possession-sequence pace",
)
completion_fig.update_layout(template="plotly_white", height=450)
completion_fig.update_yaxes(range=[0, 100])
completion_fig.show()


,pace_band,attempts,completed,games,median_pace_ft_s,mean_pass_length_ft,incomplete,Complete_pct,Incomplete_pct
0,Faster,3688,2561,34,33.49,36.7,1127,69.44,30.56
1,Middle,3688,2470,34,20.38,35.62,1218,66.97,33.03
2,Slower,3688,2541,34,12.10,35.9,1147,68.9,31.1


## 10.6. Check zone, length, type, and team context

Use the same pooled pace boundaries to compare within zone, endpoint-length band, and direct/indirect pass type. These are separate descriptive splits, not a fully adjusted model. Missing length remains an unknown group. Keep attempt and game counts when reviewing team profiles; sparse groups are not reliable rankings.


In [6]:
context_tables = {}
for context in ["event_team_zone", "length_band", "pass_type"]:
    context_passes = analysis_passes.copy()
    context_passes[context] = context_passes[context].astype("string").fillna("Unknown")
    context_tables[context] = summarize_completion(context_passes, [context, "pace_band"])
    print(context)
    display(context_tables[context].round(2))

zone_plot = context_tables["event_team_zone"]
zone_fig = px.bar(
    zone_plot, x="pace_band", y="Complete_pct", color="pace_band", facet_col="event_team_zone",
    category_orders={"pace_band": PACE_ORDER, "event_team_zone": ["DZ", "NZ", "OZ", "Unknown"]},
    color_discrete_map={"Slower": "#93c5fd", "Middle": "#3b82f6", "Faster": "#1d4ed8"},
    hover_data=["attempts", "games"],
    labels={"pace_band": "Preceding sequence pace", "Complete_pct": "Pass completion (%)"},
    title="Pass completion by zone | 5v5",
)
zone_fig.update_layout(template="plotly_white", height=450, showlegend=False)
zone_fig.update_yaxes(range=[0, 100])
zone_fig.show()
team_summary = summarize_completion(analysis_passes, ["team_id", "team_name", "pace_band"])
display(team_summary.round(2))


event_team_zone


,event_team_zone,pace_band,attempts,completed,games,median_pace_ft_s,mean_pass_length_ft,incomplete,Complete_pct,Incomplete_pct
0,DZ,Faster,1754,1246,34,36.76,40.77,508,71.04,28.96
1,DZ,Middle,1171,843,34,20.32,40.18,328,71.99,28.01
2,DZ,Slower,1710,1193,34,11.26,40.26,517,69.77,30.23
3,NZ,Faster,686,490,34,35.64,31.27,196,71.43,28.57
4,NZ,Middle,468,328,34,20.57,34.53,140,70.09,29.91
5,NZ,Slower,478,359,34,11.76,31.11,119,75.1,24.9
6,OZ,Faster,1248,825,34,29.68,33.95,423,66.11,33.89
7,OZ,Middle,2049,1299,34,20.36,33.27,750,63.4,36.6
8,OZ,Slower,1500,989,34,13.07,32.46,511,65.93,34.07


length_band


,length_band,pace_band,attempts,completed,games,median_pace_ft_s,mean_pass_length_ft,incomplete,Complete_pct,Incomplete_pct
0,0-20 ft,Faster,718,514,34,33.51,13.09,204,71.59,28.41
1,0-20 ft,Middle,793,498,34,20.45,13.1,295,62.8,37.2
2,0-20 ft,Slower,769,511,34,12.02,12.72,258,66.45,33.55
3,20-40 ft,Faster,1553,1056,34,33.51,29.78,497,68.0,32.0
4,20-40 ft,Middle,1553,1025,34,20.36,29.84,528,66.0,34.0
5,20-40 ft,Slower,1589,1114,34,12.02,29.69,475,70.11,29.89
6,Over 40 ft,Faster,1417,991,34,33.26,56.24,426,69.94,30.06
7,Over 40 ft,Middle,1342,947,34,20.38,55.62,395,70.57,29.43
8,Over 40 ft,Slower,1330,916,34,12.17,56.73,414,68.87,31.13


pass_type


,pass_type,pace_band,attempts,completed,games,median_pace_ft_s,mean_pass_length_ft,incomplete,Complete_pct,Incomplete_pct
0,Direct,Faster,2546,1803,34,33.44,35.07,743,70.82,29.18
1,Direct,Middle,2470,1654,34,20.37,33.87,816,66.96,33.04
2,Direct,Slower,2354,1657,34,12.16,33.6,697,70.39,29.61
3,Indirect,Faster,1142,758,34,33.59,40.32,384,66.37,33.63
4,Indirect,Middle,1218,816,34,20.40,39.17,402,67.0,33.0
5,Indirect,Slower,1334,884,34,11.99,39.97,450,66.27,33.73


,team_id,team_name,pace_band,attempts,completed,games,median_pace_ft_s,mean_pass_length_ft,incomplete,Complete_pct,Incomplete_pct
0,1,Boston Pride,Faster,401,261,7,34.75,37.87,140,65.09,34.91
1,1,Boston Pride,Middle,357,226,7,20.46,35.84,131,63.31,36.69
2,1,Boston Pride,Slower,317,225,7,12.10,36.88,92,70.98,29.02
3,2,Buffalo Beauts,Faster,224,133,6,36.68,33.4,91,59.38,40.62
4,2,Buffalo Beauts,Middle,271,143,6,19.96,32.5,128,52.77,47.23
5,2,Buffalo Beauts,Slower,311,191,6,11.75,36.03,120,61.41,38.59
6,3,Connecticut Whale,Faster,207,131,4,32.35,35.66,76,63.29,36.71
7,3,Connecticut Whale,Middle,203,118,4,20.31,31.97,85,58.13,41.87
8,3,Connecticut Whale,Slower,214,124,4,11.77,33.82,90,57.94,42.06
9,4,Metropolitan Riveters,Faster,167,106,3,34.43,38.08,61,63.47,36.53


## 10.7. Review individual attempts

Inspect failed passes at faster preceding pace using event and possession IDs. Review actual sequence context, intended versus actual endpoints, support, and pass selection. Check consistency across games and sensitivity to inferred possession and modeled timing before recommending a change in pace.

If completion stays flat across pace bands, report that result: higher preceding pace does not show a clear completion penalty in this sample. If differences appear, first determine whether context and selection explain them.


In [7]:
REVIEW_PACE_BAND = "Faster"
REVIEW_COMPLETE = False
review_passes = analysis_passes.loc[
    analysis_passes["pace_band"].eq(REVIEW_PACE_BAND)
    & analysis_passes["complete"].eq(REVIEW_COMPLETE)
]
display(review_passes[[
    "game_id", "game_date", "team_name", "player_name", "period", "clock_seconds",
    "event_id", "possession_id", "complete", "preceding_pace_ft_s",
    "preceding_movement_seconds", "preceding_transitions", "event_team_zone",
    "pass_type", "pass_length_ft",
]].head(20))


,game_id,game_date,team_name,player_name,period,clock_seconds,event_id,possession_id,complete,preceding_pace_ft_s,preceding_movement_seconds,preceding_transitions,event_team_zone,pass_type,pass_length_ft
14,3,2021-01-23,Minnesota Whitecaps,Haylea Schmid,1,1121.0,53,11,False,28.778847,5.500000,6,NZ,Direct,26.248809
31,3,2021-01-23,Minnesota Whitecaps,Winny Brodt-Brown,1,1052.0,107,22,False,127.906216,0.500000,2,DZ,Indirect,50.0
33,3,2021-01-23,Boston Pride,Meghara McManus,1,1040.0,112,23,False,99.317672,0.500000,2,NZ,Direct,49.091751
40,3,2021-01-23,Boston Pride,Mallory Souliotis,1,1013.0,131,27,False,59.093147,0.500000,2,DZ,Direct,19.235384
53,3,2021-01-23,Minnesota Whitecaps,Audra Richards,1,965.0,169,30,False,24.387024,6.000000,6,OZ,Direct,37.64306
60,3,2021-01-23,Boston Pride,Lexie Laing,1,910.0,204,38,False,139.269523,0.500000,2,OZ,Indirect,55.758407
81,3,2021-01-23,Boston Pride,Jenna Rheault,1,808.0,278,54,False,34.682554,1.500000,5,OZ,Indirect,50.009999
101,3,2021-01-23,Minnesota Whitecaps,Jonna Curtis,1,434.0,517,108,False,27.577819,6.500000,5,OZ,Direct,34.014703
109,3,2021-01-23,Boston Pride,Christina Putigna,1,385.0,551,114,False,36.723083,2.500000,4,DZ,Direct,25.96151
114,3,2021-01-23,Minnesota Whitecaps,Emma Stauber,1,335.0,584,120,False,25.544080,2.000000,1,NZ,Indirect,32.015621
